# PROJECT SENTINEL — Master Runner
Run cells top-to-bottom:
1. **Setup** — imports + directories
2. **Data & Features** — fetch all data, compute feature matrices (Notebook 01)
3. **Train Models** — adaptive labeling + XGBoost + validity gates (Notebook 02 Section A)
4. **Start Scheduler** — hourly inference + Telegram + 7am report (Notebook 02 Section C)

## 0 — Requirements Check
Run this cell first. It checks which packages are missing and installs only those.

In [2]:
import sys

In [3]:
import subprocess, sys, importlib.util

# pip name → import name (where they differ)
PACKAGES = {
    'ccxt':          'ccxt',
    'yfinance':      'yfinance',
    'pandas':        'pandas',
    'numpy':         'numpy',
    'xgboost':       'xgboost',
    'scikit-learn':  'sklearn',
    'pandas-ta':     'pandas_ta',
    'shap':          'shap',
    'apscheduler':   'apscheduler',
    'requests':      'requests',
    'pyarrow':       'pyarrow',
    'matplotlib':    'matplotlib',
    'seaborn':       'seaborn',
    'python-dotenv': 'dotenv',
}

missing = [pip for pip, imp in PACKAGES.items() if importlib.util.find_spec(imp) is None]

if not missing:
    print('✅ All packages already installed. Proceed to cell 1.')
else:
    print(f'📦 Installing {len(missing)} package(s): {missing}\n')

    # Stream pip output live so you can see progress
    proc = subprocess.Popen(
        [sys.executable, '-m', 'pip', 'install'] + missing,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

    if proc.returncode != 0:
        raise RuntimeError('Some packages failed to install. See errors above.')

    print('\n✅ Installation complete. Restart the kernel, then re-run from this cell.')

✅ All packages already installed. Proceed to cell 1.


## 1 — Setup

In [4]:
import os, sys, logging
from datetime import datetime, timezone, timedelta
import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import config
from utils.data_utils    import update_all_tickers, load_base_csv
from utils.features      import build_all_features, compute_anchor_features, compute_macro_risk_state
from utils.labeling      import generate_labels, find_optimal_label_params
from utils.model_utils   import (train_xgboost, evaluate_model, check_validity,
                                  get_shap_drivers, save_model, load_model, list_valid_models)
from utils.signal_manager import (is_new_signal, register_signal, check_signal_aging,
                                   check_open_signals_status, get_active_signals,
                                   get_archived_signals, recalc_open_signal_qty)
from utils.telegram_utils import send_signal, send_morning_report, send_error_alert

for d in [config.DATA_BASE, os.path.join(config.DATA_BASE,'crypto'),
          os.path.join(config.DATA_BASE,'macro'), config.DATA_WORKING,
          config.MODELS_DIR, config.STATE_DIR, config.LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(config.LOGS_DIR, 'sentinel.log')),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
print('Setup complete.')

Setup complete.


## 2 — Data & Features (Notebook 01)

In [5]:
print('=== Updating all tickers ===')
all_dfs    = update_all_tickers()
crypto_dfs = {k: v for k, v in all_dfs.items() if '/' in k}
macro_dfs  = {k: v for k, v in all_dfs.items() if '/' not in k}

# Summary
for t, df in all_dfs.items():
    if df is not None and not df.empty:
        print(f'  {t:20s}  {len(df):6d} rows  last={str(df["timestamp"].max())[:16]}')

print('\n=== Building feature matrices ===')
feature_dfs = build_all_features(crypto_dfs, macro_dfs)

for ticker, feat in feature_dfs.items():
    safe = ticker.replace('/', '_')
    path = os.path.join(config.DATA_WORKING, f'{safe}_features.parquet')
    feat.to_parquet(path, index=False)
    print(f'  {ticker:20s}  {feat.shape[0]} rows × {feat.shape[1]} cols')

print('\nData & features ready.')

=== Updating all tickers ===


2026-04-15 11:06:59,792 INFO [BTC/USDT] +13 candles -> 17492 total raw rows
--- Logging error ---
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3568.0_x64__qbz5n2kfra8p0\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3568.0_x64__qbz5n2kfra8p0\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 67: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mylai\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packa

  BTC/USDT               17524 rows  last=2026-04-15 03:00
  ETH/USDT               17524 rows  last=2026-04-15 03:00
  XRP/USDT               17524 rows  last=2026-04-15 03:00
  SOL/USDT               17524 rows  last=2026-04-15 03:00
  LTC/USDT               17524 rows  last=2026-04-15 03:00
  ADA/USDT               17524 rows  last=2026-04-15 03:00
  AAVE/USDT              17524 rows  last=2026-04-15 03:00
  LINK/USDT              17524 rows  last=2026-04-15 03:00
  AVAX/USDT              17524 rows  last=2026-04-15 03:00
  TRX/USDT               17524 rows  last=2026-04-15 03:00
  FIL/USDT               17524 rows  last=2026-04-15 03:00
  BCH/USDT               17524 rows  last=2026-04-15 03:00
  ZEC/USDT               17524 rows  last=2026-04-15 03:00
  SPX                     3474 rows  last=2026-04-14 19:30
  QQQ                     3474 rows  last=2026-04-14 19:30
  GLD                    11440 rows  last=2026-04-15 02:00
  TLT                     3474 rows  last=2026-04-14 19:

In [6]:
from utils.data_utils import check_exchange_connectivity

# ── 1. Connectivity check ─────────────────────────────────────────────────────
print('=== Checking connectivity ===')
ok, msg = check_exchange_connectivity(config.EXCHANGE_SPOT)
if ok:
    print(f'  ✅ Binance: {msg}')
else:
    print(f'  ❌ Binance unreachable: {msg}')
    print('  Suggestions:')
    print('    • Check your internet connection')
    print('    • Try a VPN if Binance is geo-restricted in your region')
    print('    • Or change config.EXCHANGE_SPOT to another ccxt exchange (e.g. "kucoin")')
    raise ConnectionError('Cannot reach exchange. Fix connectivity before continuing.')

# ── 2. Fetch & update all tickers (Layer 0 raw CSV + Layer 1 snapshot) ───────
print('\n=== Updating all tickers ===')
print('  (First run: downloading ~729 days of 1h data from yfinance — may take ~2 min)\n')
all_dfs    = update_all_tickers()
crypto_dfs = {k: v for k, v in all_dfs.items() if '/' in k}
macro_dfs  = {k: v for k, v in all_dfs.items() if '/' not in k}

for t, df in all_dfs.items():
    if df is not None and not df.empty:
        status = f'{len(df):6d} rows  last={str(df["timestamp"].max())[:16]}'
        mark   = '✅' if len(df) >= 500 else '⚠️ '
    else:
        status = 'no data'
        mark   = '❌'
    print(f'  {mark} {t:20s}  {status}')

# ── 3. Build feature matrices (requires BTC + ETH anchor data) ───────────────
print('\n=== Building feature matrices ===')
missing_anchors = [t for t in ['BTC/USDT', 'ETH/USDT'] if t not in crypto_dfs]
if missing_anchors:
    print(f'  ❌ Cannot build features — missing anchor data: {missing_anchors}')
    print('     Re-run this cell after connectivity is confirmed.')
    feature_dfs = {}
else:
    feature_dfs = build_all_features(crypto_dfs, macro_dfs)
    for ticker, feat in feature_dfs.items():
        safe = ticker.replace('/', '_')
        path = os.path.join(config.DATA_WORKING, f'{safe}_features.parquet')
        feat.to_parquet(path, index=False)
        nulls = feat.select_dtypes('number').isnull().sum().sum()
        print(f'  ✅ {ticker:20s}  {feat.shape[0]} rows × {feat.shape[1]} cols  (nulls: {int(nulls)})')
    print(f'\n  Feature matrices saved to {config.DATA_WORKING}')

2026-04-15 11:07:18,325 INFO [BTC/USDT] No new candles.


=== Checking connectivity ===
  ✅ Binance: binance reachable (4309 markets loaded)

=== Updating all tickers ===
  (First run: downloading ~729 days of 1h data from yfinance — may take ~2 min)



2026-04-15 11:07:18,570 INFO [ETH/USDT] No new candles.
2026-04-15 11:07:18,818 INFO [XRP/USDT] No new candles.
2026-04-15 11:07:19,067 INFO [SOL/USDT] No new candles.
2026-04-15 11:07:19,313 INFO [LTC/USDT] No new candles.
2026-04-15 11:07:19,560 INFO [ADA/USDT] No new candles.
2026-04-15 11:07:19,993 INFO [AAVE/USDT] No new candles.
2026-04-15 11:07:20,253 INFO [LINK/USDT] No new candles.
2026-04-15 11:07:20,500 INFO [AVAX/USDT] No new candles.
2026-04-15 11:07:20,771 INFO [TRX/USDT] No new candles.
2026-04-15 11:07:21,044 INFO [FIL/USDT] No new candles.
2026-04-15 11:07:21,311 INFO [BCH/USDT] No new candles.
2026-04-15 11:07:21,569 INFO [ZEC/USDT] No new candles.
2026-04-15 11:07:22,107 INFO [SPX] No new candles.
2026-04-15 11:07:22,619 INFO [QQQ] No new candles.
2026-04-15 11:07:23,605 ERROR $GC=F: possibly delisted; no price data found  (1h 2026-04-15 -> 2026-04-16) (Yahoo error = "Data doesn't exist for startDate = 1776225600, endDate = 1776312000")
2026-04-15 11:07:23,626 ERROR 

  ✅ BTC/USDT               17524 rows  last=2026-04-15 03:00
  ✅ ETH/USDT               17524 rows  last=2026-04-15 03:00
  ✅ XRP/USDT               17524 rows  last=2026-04-15 03:00
  ✅ SOL/USDT               17524 rows  last=2026-04-15 03:00
  ✅ LTC/USDT               17524 rows  last=2026-04-15 03:00
  ✅ ADA/USDT               17524 rows  last=2026-04-15 03:00
  ✅ AAVE/USDT              17524 rows  last=2026-04-15 03:00
  ✅ LINK/USDT              17524 rows  last=2026-04-15 03:00
  ✅ AVAX/USDT              17524 rows  last=2026-04-15 03:00
  ✅ TRX/USDT               17524 rows  last=2026-04-15 03:00
  ✅ FIL/USDT               17524 rows  last=2026-04-15 03:00
  ✅ BCH/USDT               17524 rows  last=2026-04-15 03:00
  ✅ ZEC/USDT               17524 rows  last=2026-04-15 03:00
  ✅ SPX                     3474 rows  last=2026-04-14 19:30
  ✅ QQQ                     3474 rows  last=2026-04-14 19:30
  ✅ GLD                    11440 rows  last=2026-04-15 02:00
  ✅ TLT                 

In [7]:
def get_feature_cols(df):
    exclude = {'timestamp', 'label', 'open', 'high', 'low', 'close', 'volume'}
    return [c for c in df.columns
            if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

def _atr_norm(df):
    """ATR/close array for monetary PF calculation. None if ATR_14 not available."""
    if 'ATR_14' in df.columns and 'close' in df.columns:
        return (df['ATR_14'] / df['close'].replace(0, np.nan)).values
    return None

now       = pd.Timestamp.utcnow().tz_localize(None)
train_end = now - pd.Timedelta(days=config.TRAIN_END_DAYS)
t1s = now - pd.Timedelta(days=config.TEST1[0]);  t1e = now - pd.Timedelta(days=config.TEST1[1])
t2s = now - pd.Timedelta(days=config.TEST2[0]);  t2e = now - pd.Timedelta(days=config.TEST2[1])

# Anti-leakage: remove last LABEL_HORIZON candles from training window so their
# look-ahead doesn't extend into test1.
label_buffer = pd.Timedelta(hours=config.LABEL_HORIZON)
train_cutoff = train_end - label_buffer

print(f'Train: up to {train_cutoff.date()}  |  Test1: {t1s.date()}→{t1e.date()}  |  Test2: {t2s.date()}→{t2e.date()}')
print(f'(Label buffer: {config.LABEL_HORIZON}h removed from end of training window)\n')

training_summary = []

for ticker in config.ALTCOIN_TICKERS:
    if ticker not in feature_dfs:
        print(f'[{ticker}] No features — skipped.')
        continue

    feat      = feature_dfs[ticker].copy()
    feat_cols = get_feature_cols(feat)
    ts        = pd.to_datetime(feat['timestamp'])

    # Apply label buffer to training slice
    train_df = feat[ts <= train_cutoff]
    test1_df = feat[(ts > t1s) & (ts <= t1e)]
    test2_df = feat[(ts > t2s) & (ts <= t2e)]

    if len(train_df) < 200:
        print(f'[{ticker}] Insufficient training data ({len(train_df)} rows).')
        continue

    print(f'\n[{ticker}] Grid searching label params...')
    params = find_optimal_label_params(train_df, feat_cols, verbose=False)
    tp_pct, sl_pct, k1, k2 = params['tp_pct'], params['sl_pct'], params['k1'], params['k2']
    print(f'  Best: TP={tp_pct:.3f}  SL={sl_pct:.3f}  k1={k1}  k2={k2}  '
          f'score={params["best_score"]:.2f}  PF={params["best_pf"]:.3f}  n={params["best_n"]}')

    def add_labels(df):
        lbl = generate_labels(df, tp_pct, sl_pct, k1, k2)
        df  = df.copy()
        df['label'] = lbl.values
        return df.dropna(subset=['label'])

    train_l = add_labels(train_df)
    test1_l = add_labels(test1_df)
    test2_l = add_labels(test2_df)

    if len(train_l) < 30:
        print(f'  ❌ Too few labeled training samples ({len(train_l)}). Skipping.')
        continue

    model = train_xgboost(train_l[feat_cols], train_l['label'].astype(int))

    # Monetary PF: pass tp/sl/k params + ATR_norm array
    m1 = evaluate_model(
        model, test1_l[feat_cols], test1_l['label'].astype(int),
        tp_pct=tp_pct, sl_pct=sl_pct, k1=k1, k2=k2,
        atr_norm=_atr_norm(test1_l),
    )
    m2 = evaluate_model(
        model, test2_l[feat_cols], test2_l['label'].astype(int),
        tp_pct=tp_pct, sl_pct=sl_pct, k1=k1, k2=k2,
        atr_norm=_atr_norm(test2_l),
    )
    valid = check_validity(m1, m2)

    print(f'  Test1 PF={m1["PF"]:.3f}  n={m1["trade_count"]}  wr={m1["win_rate"]:.1%}  |  '
          f'Test2 PF={m2["PF"]:.3f}  n={m2["trade_count"]}  wr={m2["win_rate"]:.1%}  |  '
          f'{"✅ VALID" if valid else "❌ INVALID"}')

    if valid:
        meta = {
            'ticker':          ticker,
            'feature_columns': feat_cols,
            'label_params':    {'tp_pct': tp_pct, 'sl_pct': sl_pct, 'k1': k1, 'k2': k2},
            'test1':           m1,
            'test2':           m2,
            'trained_at':      datetime.now(timezone.utc).isoformat(),
            'runner_ups':      params.get('runner_ups', []),
        }
        save_model(model, ticker, meta)

    training_summary.append({
        'Ticker': ticker, 'TP%': f'{tp_pct*100:.1f}', 'SL%': f'{sl_pct*100:.1f}',
        'PF_t1': m1['PF'], 'n_t1': m1['trade_count'], 'wr_t1': f'{m1["win_rate"]:.1%}',
        'PF_t2': m2['PF'], 'n_t2': m2['trade_count'],
        'Valid':  '✅' if valid else '❌',
    })

print('\n=== Training complete ===')
pd.DataFrame(training_summary).set_index('Ticker')

Train: up to 2026-01-18  |  Test1: 2026-01-20→2026-03-15  |  Test2: 2026-03-15→2026-04-14
(Label buffer: 48h removed from end of training window)


[XRP/USDT] Grid searching label params...


C:\Users\mylai\AppData\Local\Temp\ipykernel_17256\136543846.py:12: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  now       = pd.Timestamp.utcnow().tz_localize(None)


  Best: TP=0.020  SL=0.010  k1=0.5  k2=0.2  score=inf  PF=inf  n=133
  Test1 PF=0.152  n=28  wr=7.1%  |  Test2 PF=1.048  n=27  wr=33.3%  |  ❌ INVALID

[SOL/USDT] Grid searching label params...
  Best: TP=0.020  SL=0.010  k1=0.3  k2=0.2  score=inf  PF=inf  n=131
  Test1 PF=0.972  n=379  wr=33.8%  |  Test2 PF=1.246  n=128  wr=39.1%  |  ❌ INVALID

[LTC/USDT] Grid searching label params...
  Best: TP=0.025  SL=0.010  k1=0.7  k2=0.2  score=inf  PF=inf  n=32
  Test1 PF=1.027  n=307  wr=28.0%  |  Test2 PF=0.647  n=114  wr=20.2%  |  ❌ INVALID

[ADA/USDT] Grid searching label params...
  Best: TP=0.020  SL=0.010  k1=0.3  k2=0.3  score=inf  PF=inf  n=159
  Test1 PF=0.753  n=896  wr=29.7%  |  Test2 PF=0.893  n=566  wr=32.7%  |  ❌ INVALID

[AAVE/USDT] Grid searching label params...
  Best: TP=0.020  SL=0.010  k1=0.5  k2=0.2  score=inf  PF=inf  n=56
  Test1 PF=1.181  n=39  wr=35.9%  |  Test2 PF=0.000  n=0  wr=0.0%  |  ❌ INVALID

[LINK/USDT] Grid searching label params...
  Best: TP=0.040  SL=0.010 

2026-04-15 11:17:35,950 INFO [ZEC/USDT] Model saved.


  Test1 PF=1.536  n=83  wr=34.9%  |  Test2 PF=inf  n=2  wr=100.0%  |  ✅ VALID

=== Training complete ===


,TP%,SL%,PF_t1,n_t1,wr_t1,PF_t2,n_t2,Valid
Ticker,,,,,,,,
XRP/USDT,2.0,1.0,0.1516,28,7.1%,1.0475,27,❌
SOL/USDT,2.0,1.0,0.9721,379,33.8%,1.2460,128,❌
LTC/USDT,2.5,1.0,1.0272,307,28.0%,0.6471,114,❌
ADA/USDT,2.0,1.0,0.7532,896,29.7%,0.8932,566,❌
AAVE/USDT,2.0,1.0,1.1808,39,35.9%,0.0000,0,❌
LINK/USDT,4.0,1.0,1.0210,738,22.1%,0.7886,450,❌
AVAX/USDT,2.0,1.0,0.8472,529,30.4%,0.9727,250,❌
TRX/USDT,2.5,1.0,0.0000,12,0.0%,5.1689,37,❌
FIL/USDT,3.0,1.5,0.7681,1009,25.9%,0.9468,659,❌


## 3b — Model Parameter Comparison (Runner-ups)
Shows top-3 parameter combinations per coin so you can judge:
- **High PF, few trades** → strict model (may miss opportunities)
- **Lower PF, more trades** → looser model (more exposure, lower edge per trade)

Run after training to see the comparison table.

In [8]:
# ── Runner-up model comparison ─────────────────────────────────────────────────
# Shows top-3 parameter combos per coin - best + 2 runner-ups)
# runner_ups[0] = best (same as saved model), runner_ups[1/2] = alternatives

rows = []
for ticker in config.ALTCOIN_TICKERS:
    _, meta = load_model(ticker)
    if meta is None:
        continue
    lp = meta['label_params']
    m1 = meta['test1']
    m2 = meta['test2']
    runner_ups = meta.get('runner_ups', [])

    # Best model (saved)
    rows.append({
        'Coin':  ticker,
        'Rank':  '★ Best',
        'TP%':   f"{lp['tp_pct']*100:.1f}",
        'SL%':   f"{lp['sl_pct']*100:.1f}",
        'k1':    lp['k1'],
        'k2':    lp['k2'],
        'PF_t1': m1['PF'],
        'n_t1':  m1['trade_count'],
        'PF_t2': m2['PF'],
        'n_t2':  m2['trade_count'],
        'Note':  '← saved',
    })

    # Runner-ups (indices 1 and 2; index 0 is the same as best)
    rank_labels = ['  2nd', '  3rd']
    for i, ru in enumerate(runner_ups[1:3]):
        p = ru['params']
        rows.append({
            'Coin':  '',
            'Rank':  rank_labels[i],
            'TP%':   f"{p['tp_pct']*100:.1f}",
            'SL%':   f"{p['sl_pct']*100:.1f}",
            'k1':    p['k1'],
            'k2':    p['k2'],
            'PF_t1': ru['pf'],
            'n_t1':  ru['n_trades'],
            'PF_t2': '—',
            'n_t2':  '—',
            'Note':  '',
        })

if rows:
    df_cmp = pd.DataFrame(rows).set_index('Coin')
    print('=== Top-3 parameter combos per coin ===\n')
    display(df_cmp)
else:
    print('No saved models found. Run the training cell first.')

=== Top-3 parameter combos per coin ===



,Rank,TP%,SL%,k1,k2,PF_t1,n_t1,PF_t2,n_t2,Note
Coin,,,,,,,,,,
ZEC/USDT,★ Best,3.5,1.0,0.5,0.3,1.5359,83,inf,2,← saved
,2nd,3.5,1.0,0.7,0.2,inf,59,—,—,
,3rd,4.0,1.0,0.3,0.2,inf,71,—,—,


## 3c — Active Signals Report (with SHAP Drivers)
Run this cell at any time to see current open signals + the top-5 features driving each model's prediction.

In [9]:
# ── Active signals with SHAP drivers ──────────────────────────────────────────
from utils.signal_manager import _rr

active = get_active_signals()
if not active:
    print('No active signals.')
else:
    print(f'=== Active signals: {len(active)} ===\n')
    for sig in active:
        coin    = sig['coin']
        icon    = '🟢' if sig['direction'] == 'LONG' else '🔴'
        print(f'{icon} {coin}  {sig["direction"]}')
        print(f'   Entry: {sig["entry"]:.4f}  TP: {sig["tp"]:.4f}  SL: {sig["sl"]:.4f}')
        print(f'   P(win): {sig["p_win"]:.2f}  Qty: {sig["qty"]:.4f}  Repeats: {sig["repeat_count"]}')
        print(f'   Since: {sig["first_signal_at"]}')

        # Load model + compute SHAP drivers from latest feature snapshot
        model, meta = load_model(coin)
        if model is not None and meta is not None:
            feat_path = os.path.join(
                config.DATA_WORKING,
                f"{coin.replace('/', '_')}_features.parquet"
            )
            if os.path.exists(feat_path):
                try:
                    feat_live = pd.read_parquet(feat_path)
                    feat_cols = meta['feature_columns']
                    # Use the last row with all required features present
                    valid_rows = feat_live.dropna(subset=feat_cols)
                    if not valid_rows.empty:
                        X_live   = valid_rows[feat_cols].tail(1)
                        p_latest = float(model.predict_proba(X_live)[0, 1])
                        drivers  = get_shap_drivers(model, X_live, n=5)
                        print(f'   Current P(win): {p_latest:.2f}')
                        print(f'   Top drivers: {", ".join(drivers)}')
                    else:
                        print('   (feature data unavailable for SHAP)')
                except Exception as e:
                    print(f'   (SHAP error: {e})')
            else:
                print('   (feature parquet not found — run cell 2 first)')
        print()

No active signals.


## 4 -- Start Scheduler (Section C)
Run this cell to start the hourly pipeline + 7am report.

Leave it running -- interrupt the kernel to stop.

In [ ]:
import time
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.events import EVENT_JOB_ERROR
from utils.features import compute_altcoin_features

# ── Job: hourly inference ─────────────────────────────────────────────────────
def run_hourly_pipeline():
    logger.info('=== Hourly pipeline START ===')
    try:
        dfs   = update_all_tickers()
        c_dfs = {k: v for k, v in dfs.items() if '/' in k}
        m_dfs = {k: v for k, v in dfs.items() if '/' not in k}
        if c_dfs.get('BTC/USDT') is None:
            logger.error('BTC/USDT missing — skipping inference.')
            return

        ms     = compute_macro_risk_state(m_dfs)
        btc_a  = compute_anchor_features(c_dfs['BTC/USDT'], 'BTC')
        eth_a  = compute_anchor_features(c_dfs['ETH/USDT'], 'ETH')

        current_prices = {t: float(c_dfs[t]['close'].iloc[-1])
                          for t in config.ALTCOIN_TICKERS if t in c_dfs}
        check_open_signals_status(current_prices)

        for ticker in list_valid_models():
            if ticker not in c_dfs:
                continue
            model, meta = load_model(ticker)
            if model is None:
                continue
            feat_cols = meta['feature_columns']
            lp        = meta['label_params']

            feat_all  = compute_altcoin_features(c_dfs[ticker], btc_a, eth_a, ms)
            feat_live = feat_all.dropna(subset=feat_cols).tail(1)
            if feat_live.empty:
                continue

            try:
                X_live = feat_live[feat_cols]
            except KeyError as e:
                logger.error(f'[{ticker}] Feature parity error: {e}')
                continue

            p_win   = float(model.predict_proba(X_live)[0, 1])
            close   = float(c_dfs[ticker]['close'].iloc[-1])
            atr     = float(feat_live['ATR_14'].iloc[0]) if 'ATR_14' in feat_live.columns else 0.0
            atr_n   = atr / close if close > 0 else 0.0
            drivers = get_shap_drivers(model, X_live, n=5)

            def _fire(direction, tp, sl):
                qty = config.MAX_LOSS_USDT / max(abs(close - sl), 1e-8)
                if is_new_signal(ticker, direction):
                    register_signal(ticker, direction, close, tp, sl, qty,
                                    p_win, lp['tp_pct'], lp['sl_pct'])
                    send_signal(ticker, direction, close, tp, sl, qty, p_win, drivers)
                    logger.info(f'[{ticker}] {direction} signal fired P={p_win:.2f}')
                else:
                    aging = check_signal_aging(ticker, close)
                    if aging == 'repeat':
                        active_sigs = [s for s in get_active_signals() if s['coin'] == ticker]
                        if active_sigs:
                            s = active_sigs[0]
                            send_signal(ticker, direction, s['entry'], s['tp'], s['sl'],
                                        s['qty'], p_win, drivers, is_repeat=True)

            if p_win >= config.LONG_THRESHOLD:
                tp = close * (1 + lp['tp_pct'] + lp['k1'] * atr_n)
                sl = close * (1 - lp['sl_pct'] - lp['k2'] * atr_n)
                _fire('LONG', tp, sl)
            elif p_win <= config.SHORT_THRESHOLD:          # ← fixed: was (1 - config.SHORT_THRESHOLD)
                tp = close * (1 - lp['tp_pct'] - lp['k1'] * atr_n)
                sl = close * (1 + lp['sl_pct'] + lp['k2'] * atr_n)
                _fire('SHORT', tp, sl)

    except Exception as e:
        logger.error(f'Hourly pipeline error: {e}')
        send_error_alert(f'Hourly pipeline error: {e}')
    logger.info('=== Hourly pipeline END ===')


# ── Job: retrain ──────────────────────────────────────────────────────────────
def run_retrain():
    logger.info('=== Retraining START ===')
    try:
        dfs      = update_all_tickers()
        c_dfs    = {k: v for k, v in dfs.items() if '/' in k}
        m_dfs    = {k: v for k, v in dfs.items() if '/' not in k}
        feat_dfs = build_all_features(c_dfs, m_dfs)
        for t, feat in feat_dfs.items():
            feat.to_parquet(os.path.join(config.DATA_WORKING,
                            f'{t.replace("/","_")}_features.parquet'), index=False)

        now_r  = pd.Timestamp.utcnow().tz_localize(None)
        te_raw = now_r - pd.Timedelta(days=config.TRAIN_END_DAYS)
        # Anti-leakage: remove last LABEL_HORIZON candles from training cutoff
        te     = te_raw - pd.Timedelta(hours=config.LABEL_HORIZON)
        t1s_r  = now_r - pd.Timedelta(days=config.TEST1[0])
        t1e_r  = now_r - pd.Timedelta(days=config.TEST1[1])
        t2s_r  = now_r - pd.Timedelta(days=config.TEST2[0])
        t2e_r  = now_r - pd.Timedelta(days=config.TEST2[1])

        for ticker in config.ALTCOIN_TICKERS:
            if ticker not in feat_dfs:
                continue
            feat = feat_dfs[ticker]
            fc   = get_feature_cols(feat)
            ts   = pd.to_datetime(feat['timestamp'])
            tr   = feat[ts <= te]
            t1   = feat[(ts > t1s_r) & (ts <= t1e_r)]
            t2   = feat[(ts > t2s_r) & (ts <= t2e_r)]
            if len(tr) < 200:
                continue

            p = find_optimal_label_params(tr, fc, verbose=False)
            tp_pct, sl_pct, k1, k2 = p['tp_pct'], p['sl_pct'], p['k1'], p['k2']

            def al(df):
                df = df.copy()
                df['label'] = generate_labels(df, tp_pct, sl_pct, k1, k2).values
                return df.dropna(subset=['label'])

            tr_l = al(tr)
            t1_l = al(t1)
            t2_l = al(t2)
            if len(tr_l) < 30:
                continue

            m = train_xgboost(tr_l[fc], tr_l['label'].astype(int))

            # Monetary PF: pass label params + ATR_norm
            m1 = evaluate_model(
                m, t1_l[fc], t1_l['label'].astype(int),
                tp_pct=tp_pct, sl_pct=sl_pct, k1=k1, k2=k2,
                atr_norm=_atr_norm(t1_l),
            )
            m2 = evaluate_model(
                m, t2_l[fc], t2_l['label'].astype(int),
                tp_pct=tp_pct, sl_pct=sl_pct, k1=k1, k2=k2,
                atr_norm=_atr_norm(t2_l),
            )

            if check_validity(m1, m2):
                save_model(m, ticker, {
                    'ticker':          ticker,
                    'feature_columns': fc,
                    'label_params':    {'tp_pct': tp_pct, 'sl_pct': sl_pct,
                                        'k1': k1, 'k2': k2},
                    'test1':           m1,
                    'test2':           m2,
                    'trained_at':      datetime.now(timezone.utc).isoformat(),
                    'runner_ups':      p.get('runner_ups', []),
                })
                logger.info(f'[{ticker}] Retrained OK — PF_t1={m1["PF"]:.3f}  PF_t2={m2["PF"]:.3f}')
            else:
                logger.info(f'[{ticker}] Retrain invalid (PF_t1={m1["PF"]:.3f}) — old model kept.')

    except Exception as e:
        logger.error(f'Retrain error: {e}')
        send_error_alert(f'Retrain error: {e}')
    logger.info('=== Retraining END ===')


# ── Job: morning report ───────────────────────────────────────────────────────
def send_morning_report_job():
    try:
        dfs = update_all_tickers()
        prices = {t: float(dfs[t]['close'].iloc[-1])
                  for t in config.ALTCOIN_TICKERS if t in dfs and not dfs[t].empty}
        since = datetime.now(timezone.utc) - timedelta(hours=24)
        send_morning_report(get_active_signals(), get_archived_signals(since=since), prices)
    except Exception as e:
        logger.error(f'Morning report error: {e}')


# ── Start ─────────────────────────────────────────────────────────────────────
scheduler = BackgroundScheduler(timezone='UTC')
scheduler.add_listener(lambda ev: logger.error(f'Job error: {ev.exception}'), EVENT_JOB_ERROR)

scheduler.add_job(run_hourly_pipeline,    'interval', minutes=config.SCHEDULER_INTERVAL_MIN,
                  id='hourly', misfire_grace_time=300, coalesce=True)
scheduler.add_job(run_retrain,            'interval', hours=config.RETRAIN_INTERVAL_HOURS,
                  id='retrain', misfire_grace_time=1800, coalesce=True)
scheduler.add_job(send_morning_report_job,'cron',     hour=config.MORNING_REPORT_HOUR, minute=0,
                  id='morning_report')

scheduler.start()
print(f'Scheduler running. Next run in {config.SCHEDULER_INTERVAL_MIN} min. Interrupt kernel to stop.\n')

# Run one cycle immediately so you see output right away
print('Running initial inference cycle...')
run_hourly_pipeline()

try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    scheduler.shutdown()
    print('Scheduler stopped.')

2026-04-15 11:17:36,411 INFO Adding job tentatively -- it will be properly scheduled when the scheduler starts
2026-04-15 11:17:36,413 INFO Adding job tentatively -- it will be properly scheduled when the scheduler starts
2026-04-15 11:17:36,425 INFO Adding job tentatively -- it will be properly scheduled when the scheduler starts
2026-04-15 11:17:36,426 INFO Added job "run_hourly_pipeline" to job store "default"
2026-04-15 11:17:36,428 INFO Added job "run_retrain" to job store "default"
2026-04-15 11:17:36,430 INFO Added job "send_morning_report_job" to job store "default"
2026-04-15 11:17:36,431 INFO Scheduler started
2026-04-15 11:17:36,434 INFO === Hourly pipeline START ===


Scheduler running. Next run in 60 min. Interrupt kernel to stop.

Running initial inference cycle...


2026-04-15 11:17:37,948 INFO [BTC/USDT] No new candles.
2026-04-15 11:17:38,318 INFO [ETH/USDT] No new candles.
2026-04-15 11:17:38,567 INFO [XRP/USDT] No new candles.
2026-04-15 11:17:38,825 INFO [SOL/USDT] No new candles.
2026-04-15 11:17:39,090 INFO [LTC/USDT] No new candles.
2026-04-15 11:17:39,354 INFO [ADA/USDT] No new candles.
2026-04-15 11:17:39,606 INFO [AAVE/USDT] No new candles.
2026-04-15 11:17:40,021 INFO [LINK/USDT] No new candles.
2026-04-15 11:17:40,278 INFO [AVAX/USDT] No new candles.
2026-04-15 11:17:40,536 INFO [TRX/USDT] No new candles.
2026-04-15 11:17:40,791 INFO [FIL/USDT] No new candles.
2026-04-15 11:17:41,043 INFO [BCH/USDT] No new candles.
2026-04-15 11:17:41,312 INFO [ZEC/USDT] No new candles.
2026-04-15 11:17:42,046 INFO [SPX] No new candles.
2026-04-15 11:17:42,721 INFO [QQQ] No new candles.
2026-04-15 11:17:43,898 ERROR $GC=F: possibly delisted; no price data found  (1h 2026-04-15 -> 2026-04-16) (Yahoo error = "Data doesn't exist for startDate = 177622560

In [ ]:
# try:
#     while True:
#         time.sleep(30)
# except KeyboardInterrupt:
#     scheduler.shutdown()
#     print('Scheduler stopped.')